In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
import os

class ImageNet(Dataset):
    def __init__(self, root: str, annotations_path, transform=None, p: float = 0.5, max_hint: float = 0.1):
        self.root = root
        self.transform = transform
        self.p = p
        self.max_hint = max_hint
        with open(annotations_path) as f:
            self.dataset = f.readlines()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx: int):
        path = self.dataset[idx].strip().split(" ")[0] + ".JPEG"
        path = os.path.join(self.root, path)
        image = cv2.imread(path)  # BGR
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
            if isinstance(image, torch.Tensor):
                image = image.permute(1, 2, 0).cpu().numpy() * 255  # CHW → HWC, float → 0-255
                image = image.astype(np.uint8)

        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB).astype(np.float32)
        L = lab[:, :, 0:1] / 50.0 - 1.0
        ab = (lab[:, :, 1:] - 128.0) / 110.0

        image_tensor = torch.from_numpy(image.astype(np.float32) / 255.0).permute(2, 0, 1)
        L_tensor = torch.from_numpy(L).permute(2, 0, 1)
        ab_tensor = torch.from_numpy(ab).permute(2, 0, 1)

        hint, mask = self._make_hint(ab_tensor)

        return {
            "image": image_tensor,
            "L": L_tensor,
            "ab": ab_tensor,
            "hint": hint,
            "mask": mask
        }

    def _make_hint(self, ab: torch.Tensor):
        if torch.rand(1).item() > self.p:
            return torch.zeros_like(ab), torch.zeros(1, ab.shape[1], ab.shape[2])

        hint = torch.zeros_like(ab)
        mask = torch.zeros(1, ab.shape[1], ab.shape[2])
        num_points = int(self.max_hint * ab.shape[1] * ab.shape[2])
        ys = torch.randint(0, ab.shape[1], (num_points,))
        xs = torch.randint(0, ab.shape[2], (num_points,))
        hint[:, ys, xs] = ab[:, ys, xs]
        mask[:, ys, xs] = 1.0
        return hint, mask

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class NeuralColor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(4, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),

            nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 2, 1)  # output ab
            )

    def forward(self, x):
        return self.net(x)

In [ ]:
import torch

import numpy as np

import matplotlib.pyplot as plt

import os
import random
import math
import json

def seed_everything(seed):
    # Set fixed seed for reproducibility across libraries
    random.seed(seed)                          # Python random module seed
    np.random.seed(seed)                       # NumPy seed
    torch.manual_seed(seed)                    # PyTorch CPU seed
    torch.cuda.manual_seed(seed)               # PyTorch CUDA seed for single GPU
    torch.cuda.manual_seed_all(seed)           # PyTorch CUDA seed for all GPUs if using multi-GPU
    # Ensure deterministic behavior for CUDA convolution operations
    torch.backends.cudnn.deterministic = True
    # Disable benchmark mode to prevent nondeterministic algorithm selection
    torch.backends.cudnn.benchmark = False

def init_history(keys: list[str] | None = None) -> dict[str, dict[str, list]]:
    if keys is None:
        keys = ["loss"]  # default keys

    return {phase: {key: [] for key in keys} for phase in ["train", "val"]}

def plot_history(
    history: dict[str, dict[str, list]],
    paths: dict = {},
    save: bool = True,
    root: str = "sample"
):
    os.makedirs(root, exist_ok=True)
    
    # Extract training and validation history
    train_history = history['train']
    val_history = history['val']
    
    # Generate epoch indices
    epochs = range(1, len(train_history['loss']) + 1)
    
    # Define full paths for saving plot image and history file
    history_path = os.path.join(root, paths.get("history", "history.json"))
    plot_image_path = os.path.join(root, paths.get("plot_image", "history_plot.png"))

    n_metrics = len(train_history)
    cols = 2
    rows = math.ceil(n_metrics / cols)    
    
    # Create a figure with subplots for each metric
    plt.figure(figsize=(5 * cols, 4 * rows))
    
    for i, k in enumerate(train_history):
        plt.subplot(rows, cols, i + 1)
        plt.plot(epochs, train_history[k], 'bo-', label=f'Train {k.capitalize()}')
        plt.plot(epochs, val_history[k], 'ro-', label=f'Val {k.capitalize()}')
        plt.xlabel('Epoch')
        plt.ylabel(k.capitalize())
        plt.title(f'{k.capitalize()} over Epochs')
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    
    # Save plot
    if save:
        plt.savefig(plot_image_path)
        # Save history as JSON
        with open(history_path, "w") as f:
            json.dump(history, f, indent=4)
    
    plt.show()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from tqdm import tqdm

import time
import os
import re
from typing import Optional

def denorm_lab(L, ab):
    L = L * 100.0
    ab = ab * 128.0
    return L, ab

def compute_deltaE(pred_ab, gt_ab, L):
    # scale lại
    L, pred_ab = denorm_lab(L, pred_ab)
    _, gt_ab   = denorm_lab(L, gt_ab)

    # ghép thành Lab
    pred_lab = torch.cat([L, pred_ab], dim=1)
    gt_lab   = torch.cat([L, gt_ab], dim=1)

    # deltaE
    deltaE = torch.sqrt(((pred_lab - gt_lab) ** 2).sum(dim=1))  # (B,H,W)
    
    return deltaE.mean()

def get_latest_epoch(
    root: str = 'models', 
    prefix="model_epoch_", 
    suffix=".pth"
) -> int:
    max_epoch = 0  # Initialize max epoch to 0

    # Iterate through all files in the specified folder
    for filename in os.listdir(root):
        # Check if the filename matches the pattern "model_epoch_{number}.pth"
        match = re.match(fr"{prefix}(\d+){suffix}", filename)
        if match:
            # Extract the epoch number from the filename
            epoch_num = int(match.group(1))
            # Update max_epoch if this epoch number is greater
            if epoch_num > max_epoch:
                max_epoch = epoch_num

    return max_epoch  # Return the highest epoch number found

def save_best_models(
    model: nn.Module,
    val_result: dict[str, float],
    epoch: int,
    save_paths: dict[str, str],
    root: str = "models",
    best_metrics: dict[str, float] | None = None,
) -> bool:

    os.makedirs(root, exist_ok=True)

    if best_metrics is None:
        best_metrics = {}

    updated = False

    for metric, value in val_result.items():

        if metric not in save_paths:
            continue

        save_path = os.path.join(root, save_paths[metric])

        best_val = best_metrics.get(metric, float("inf"))

        if value < best_val:
            best_metrics[metric] = value
            torch.save(model.state_dict(), save_path)
            print(f"✅✅Best {metric} model saved at epoch {epoch+1:02d}")
            updated = True

    return updated

def save_epoch_model(
    model: nn.Module,
    epoch: int,
    root: str="models"
):
    # Save the model for the current epoch (regardless of performance)
    path = os.path.join(root, f"model_epoch_{epoch + 1:02d}.pth")
    torch.save(model.state_dict(), path)
    print(f"Model for epoch {epoch + 1:02d} saved.")


def forward_pass(loop, model, device):
    images = loop["image"].to(device, non_blocking=True)
    L = loop["L"].to(device, non_blocking=True)
    ab = loop["ab"].to(device, non_blocking=True)
    hint = loop["hint"].to(device, non_blocking=True)
    mask = loop["mask"].to(device, non_blocking=True)

    x = torch.cat([L, hint, mask], dim=1)
    outputs = model(x)

    return images, L, ab, outputs


def compute_metrics(outputs, ab, L, loss_fn):
    loss = loss_fn(outputs, ab)
    deltaE = compute_deltaE(outputs, ab, L)
    return loss, deltaE


def update_stats(total_loss: float, total_deltaE: float, total_samples: int, 
                 loss, deltaE, batch_size):
    total_loss += loss.item() * batch_size
    total_deltaE += deltaE.item() * batch_size
    total_samples += batch_size
    return total_loss, total_deltaE, total_samples


def finalize_stats(total_loss: float, total_deltaE: float, total_samples: int):
    return {
        "loss": total_loss / total_samples,
        "deltaE": total_deltaE / total_samples,
    }
    
def train(model: nn.Module, 
          train_loader: DataLoader, 
          optimizer: torch.optim.Adam, 
          device: torch.device, 
          criterion: dict):
    model.train()

    total_loss = 0.0
    total_deltaE = 0.0
    total_samples = 0

    loss_fn = criterion["mse_loss"]

    loops = tqdm(train_loader, desc="Training", leave=False)

    
    for i, loop in enumerate(loops):
        if i >= 5:
            break
        images, L, ab, outputs = forward_pass(loop, model, device)

        optimizer.zero_grad()

        loss = loss_fn(outputs, ab)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            _, deltaE = compute_metrics(outputs, ab, L, loss_fn)

        batch_size = images.size(0)

        total_loss, total_deltaE, total_samples = update_stats(
            total_loss, total_deltaE, total_samples,
            loss, deltaE, batch_size
        )

        loops.set_postfix(loss=loss.item(), deltaE=deltaE.item())

    return finalize_stats(total_loss, total_deltaE, total_samples)

@torch.no_grad()
def evaluate(model: nn.Module, 
             data_loader: DataLoader, 
             device: torch.device, 
             criterion: dict):
    model.eval()

    total_loss = 0.0
    total_deltaE = 0.0
    total_samples = 0

    loss_fn = criterion["mse_loss"]

    loops = tqdm(data_loader, desc="Evaluating", leave=False)

    for i, loop in enumerate(loops):
        if i >= 5:
            break
        images, L, ab, outputs = forward_pass(loop, model, device)

        loss, deltaE = compute_metrics(outputs, ab, L, loss_fn)

        batch_size = images.size(0)

        total_loss, total_deltaE, total_samples = update_stats(
            total_loss, total_deltaE, total_samples,
            loss, deltaE, batch_size
        )

        loops.set_postfix(loss=loss.item(), deltaE=deltaE.item())

    return finalize_stats(total_loss, total_deltaE, total_samples)

def fit(model: nn.Module, optimizer: torch.optim.Adam,
        device: torch.device, epochs: int,
        loader: dict["str", DataLoader], criterion: dict = {},
        gamma: float = 0.5, patience: int = 6,
        save_paths: dict[str, str] = {
            "loss":"best_loss_model.pth",
            "deltaE":"best_delta_model.pth"
        },
        history: dict[str, dict[str, list]] = {},
        roots: dict[str, str] = {"model": "models", "sample": "sample"},
        best_metrics: dict = {"loss": float("inf")},
        ):

    start_epoch = get_latest_epoch(roots["model"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=gamma, patience=patience//2)
    early_stop_counter = 0

    for epoch in range(start_epoch, start_epoch + epochs):
        start_time: float = time.time()
        print('=' * 25 + f'Epoch {epoch + 1:02d}/{start_epoch + epochs:02d}' + '=' * 25)
        train_loader = loader["train"]
        train_result: dict[str, dict[str, float]] = train(model, train_loader, optimizer, device, criterion)
        if "val" in loader.keys():
            val_loader = loader["val"]
            val_result: Optional[dict[str, dict[str, float]]] = evaluate(model, val_loader, device, criterion)
        else:
            val_result = None
        keys = list(history['train'].keys())
        
        for k in keys:
            # ---- train history ----
            history['train'][k].append(train_result[k])

            # ---- Validation history ----
            if "val" in loader.keys() and val_result is not None:
                history['val'][k].append(val_result[k])
            else:
                history['val'][k].append(-1)
        if "val" in loader.keys() and val_result is not None:
            if save_best_models(model, val_result, epoch, save_paths, roots["model"], best_metrics):
                early_stop_counter = 0
            else:
                early_stop_counter += 1
                print(f'Stopping counter {early_stop_counter}/{patience}')
        save_epoch_model(model, epoch, roots["model"])
    
        minutes, seconds = divmod(time.time() - start_time, 60)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch time: {int(minutes):02d}:{int(seconds):02d}"
            f"LR: {current_lr:.6f}")

        # ===== LOSS =====
        if "val" in loader.keys() and val_result is not None:
            print(
                f"Train Loss: {train_result['loss']:.4f} | "
                f"Val Loss: {val_result['loss']:.4f} | "
            )
            print(
                # Overal Delta E
                f"Train ΔE: {train_result['deltaE']:.4f} | "
                f"Val ΔE: {val_result['deltaE']:.4f} | "
            )
        else:
            print(
                f"Train Loss: {train_result['loss']:.4f} | "
            )
            print(
                # Overal Delta E
                f"Train ΔE: {train_result['rmse']:.4f} | "
            )

            
        if "val" in loader.keys() and val_result is not None:
            if early_stop_counter >= patience:
                print("❌❌Early stopping triggered")
                scheduler.step(val_result['rmse'])
                return history

    return history

In [ ]:
import shutil
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, random_split, DistributedSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2

config = {
    "seed": 42,
    "ROOT": "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC",
    
    "MODEL_FOLDER": "models",
    "SAMPLE_FOLDER": "sample",
    "plot_image_path": "plot_image.png",
    "history_path": "history.json",
    "model_path": "models/best_delta_E_model.pth",
    "BATCH_SIZE": 128,
    "EPOCHS": 10,
    "IMG_SIZE": 224,
    "LR": 3e-4,
    "ratio": {"train": 0.8, "val": 0.2}
}

criterion = {
    "mse_loss": F.mse_loss,
}


seed_everything(config["seed"])  # Set random seed for reproducibility

# Select device: use GPU if available, otherwise CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Number of CPU cores for data loading parallelism
num_workers = os.cpu_count()

print(f"Device: {device}")
print(f"Number of CPU cores: {num_workers}")

# Flag to indicate whether to clear old data before running
clear = True

if clear:
    for root in (config["MODEL_FOLDER"], config["SAMPLE_FOLDER"]):
        try:
            shutil.rmtree(root)
        except Exception:
            pass

        try:
            os.makedirs(root, exist_ok=True)
        except Exception:
            pass


def build_transforms(IMG_SIZE: int = 224):
    train_transform = A.Compose([
        A.Resize(height=IMG_SIZE, width=IMG_SIZE),
        # Biến đổi hình học
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=0.5),
        # A.RandomResizedCrop(height=IMG_SIZE, width=IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0),

        # Biến đổi màu/ảnh grayscale (L channel)
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.GaussNoise(
            std_range=(5/255, 20/255),
            mean_range=(0.0, 0.0),
            per_channel=True,
            noise_scale_factor=1.0,
            p=0.3
        ),
        
        # Color jitter / ab perturb: dùng HueSaturationValue như proxy cho ab
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.3),

        # To tensor
        ToTensorV2()
    ])

    # Validation transform: chỉ resize/crop, không augment
    val_transform = A.Compose([
        A.Resize(height=IMG_SIZE, width=IMG_SIZE),
        ToTensorV2()
    ])
    

    return train_transform, val_transform


train_transform, val_transform = build_transforms(config["IMG_SIZE"])
train_annotations_path = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/ImageSets/CLS-LOC/train_cls.txt"
val_annotations_path = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/ImageSets/CLS-LOC/val.txt"
test_annotations_path = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/ImageSets/CLS-LOC/test.txt"
val_dataset = ImageNet(os.path.join(config["ROOT"], "val"), val_annotations_path, val_transform)
test_dataset = ImageNet(os.path.join(config["ROOT"], "test"), test_annotations_path ,val_transform)
train_dataset = ImageNet(os.path.join(config["ROOT"], "train"), train_annotations_path ,train_transform)

train_loader = DataLoader(train_dataset, batch_size=config["BATCH_SIZE"], shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config["BATCH_SIZE"], shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config["BATCH_SIZE"], shuffle=False, num_workers=num_workers, pin_memory=True)

best_metrics = {
    "loss": float("inf"),
    "deltaE": float("inf")
}
default_keys = ['loss', 'deltaE']
history = init_history(default_keys)

model = NeuralColor().to(device)
model = torch.nn.DataParallel(model) 
model.cuda()

loader = {"train": train_loader, "val": val_loader}

optimizer = torch.optim.AdamW(
    [{"params": model.net.parameters(), "lr": config["LR"]}],
    weight_decay=0.01)

In [ ]:
history = fit(
    model=model, 
    optimizer=optimizer, 
    device=device, 
    epochs=config["EPOCHS"], 
    criterion=criterion, 
    loader=loader,
    history=history, 
    best_metrics=best_metrics,
)

In [ ]:

plot_history(history, {"plot_image": config["plot_image_path"], "history": config["history_path"]}, root=config["SAMPLE_FOLDER"])